# 10 — Item-level multimodal rank fusion

**Milestone 3a.** Savoroute combines BM25, text-vector, and image-vector *rank lists* into one ranking of catalog items. A restaurant or recipe can have many media records, but it appears only once in the final result.

The notebook uses bundled fictional records and a deterministic **test encoder**; it does not measure CLIP quality. The package performs all retrieval and ranking; the notebook only calls its public APIs.

## 1. Prepare the catalog and media

The image fixture is normalized through Savoroute's existing ingestion pipeline. Files are written to a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from savoroute.data.catalog import load_catalog
from savoroute.media.ingestion import prepare_image_batch, write_image_batch

root = Path.cwd()
assert (root / "data/sample/catalog.json").exists(), "Run JupyterLab from the project root"
sample = root / "data/sample"
workspace = TemporaryDirectory(prefix="savoroute-fusion-")
media_dir = Path(workspace.name) / "media"
write_image_batch(prepare_image_batch(sample / "image_manifest.json",
                                      load_catalog(sample / "catalog.json")), media_dir)
catalog = load_catalog(media_dir / "catalog.json")
print("Items:", len(catalog.items), "Media:", len(catalog.media))

## 2. Build an index with the deterministic test encoder

This encoder assigns two-dimensional vectors based on keywords and image color. It verifies the retrieval workflow without downloading pretrained weights or claiming semantic understanding.

In [ ]:
from PIL import Image, ImageStat
from savoroute.retrieval.vector import build_vector_index, validate_catalog

class DemoEncoder:
    model_id = "offline-fusion-demo"
    embedding_space_id = "offline-fusion-demo@1"

    def embed_texts(self, texts):
        return [[0.0, 1.0] if "mushroom" in text.casefold() else [1.0, 0.0]
                for text in texts]

    def embed_images(self, paths):
        vectors = []
        for path in paths:
            with Image.open(path) as image:
                green = ImageStat.Stat(image.convert("RGB")).mean[1]
            vectors.append([0.0, 1.0] if green > 190 else [1.0, 0.0])
        return vectors

encoder = DemoEncoder()
index = build_vector_index(catalog, media_dir, encoder)
validate_catalog(index, catalog, catalog_dir=media_dir)
print("Indexed:", len(index.records), "vectors; space:", index.embedding_space_id)

## 3. Why not average BM25 and cosine scores?

BM25 and cosine similarity operate on different scales. Even cosine values from different models need not have comparable relevance semantics. We therefore fuse *ranks* instead of their raw scores.

For a candidate item \(i\), weighted reciprocal-rank fusion is

\[S(i) = \sum_{s \in A(i)} \frac{w_s / \sum_{t\in A} w_t}{k_0 + r_s(i)}.\]

Here \(A\) contains active (nonempty) sources, \(A(i)\) contains sources that retrieved item \(i\), and \(r_s(i)\) is its 1-based source rank. We use \(k_0 = 60\). The RRF score is **not** a relevance probability.

In [ ]:
from savoroute.retrieval.fusion import FusionWeights, search_fused
from savoroute.retrieval.lexical import SearchFilters, search_catalog

query = "mushroom pasta"
bm25 = search_catalog(catalog.items, query, SearchFilters(kind="recipe"), limit=5)
print("BM25:", [(h.item.item_id, round(h.score, 3)) for h in bm25])
combined = search_fused(catalog, query, index=index, encoder=encoder,
                        catalog_dir=media_dir, filters=SearchFilters(kind="recipe"),
                        pool_size=12, limit=5)
for hit in combined:
    print(hit.item.item_id, hit.item.title, round(hit.score, 6),
          [(e.source, e.rank, e.record_id) for e in hit.evidence])

## 4. Verify hard filters and one result per item

Filters are applied before source ranking. Metadata that does not satisfy a requested constraint cannot enter any ranking. An item's image or text cannot bypass a restaurant/recipe or dietary filter.

In [ ]:
filtered = search_fused(catalog, "noodles", index=index, encoder=encoder,
                      catalog_dir=media_dir,
                      filters=SearchFilters(kind="restaurant", location="Sample City",
                                            max_price_tier=2), limit=10)
print([(hit.item.item_id, hit.item.kind, hit.item.location) for hit in filtered])
assert all(hit.item.kind == "restaurant" for hit in filtered)
assert len({hit.item.item_id for hit in filtered}) == len(filtered)

## 5. Weight the sources

The source weights describe an explicit retrieval policy; they are not trained in this notebook. With a zero image weight, no image contribution is included. The relative rankings can change with weights and candidate-pool size.

In [ ]:
for name, weights in [("balanced", FusionWeights(1, 1, 1)),
                      ("text-first", FusionWeights(3, 2, 0)),
                      ("image-first", FusionWeights(1, 1, 4))]:
    ranked = search_fused(catalog, "mushroom pasta", index=index, encoder=encoder,
                         catalog_dir=media_dir, weights=weights,
                         filters=SearchFilters(kind="recipe"), limit=3)
    print(name, [row.item.item_id for row in ranked])

## 6. Evaluate the same implementation used by the CLI

The retrieval benchmark contains synthetic item-level judgments and negative queries. Positive-query Recall@k, MRR@k, and nDCG@k are reported separately from the correct-empty rate.

**Important:** cosine search always returns nearest neighbors if eligible vectors exist. Without a validated relevance threshold, the hybrid system may return results for queries with no relevant item. Synthetic scores cannot justify production claims.

In [ ]:
from savoroute.evaluation.fusion import evaluate_fusion
from savoroute.evaluation.retrieval import evaluate_search, load_benchmark

judgments = load_benchmark(sample / "relevance_benchmark.json")
lexical = evaluate_search(catalog.items, judgments, at_k=5, method="bm25")
hybrid = evaluate_fusion(catalog, judgments, index=index, encoder=encoder,
                         catalog_dir=media_dir, at_k=5, pool_size=12)
for label, summary in [("BM25", lexical), ("RRF (test encoder)", hybrid)]:
    print(label, "Recall@5:", summary.mean_recall_at_k if label == "BM25" else summary.recall_at_k,
          "MRR@5:", summary.mrr_at_k,
          "Correct-empty:", summary.correct_empty_rate)

## 7. API and CLI parity

`DiscoveryService.rank()` calls the same function. For model-backed local runs, build a real CLIP index as in notebook 09, then use:

```bash
savoroute rank search "mushroom pasta" \
  --catalog data/generated/m2b/media/catalog.json \
  --index data/generated/m2b/vectors.json \
  --kind recipe --json

savoroute rank evaluate \
  --catalog data/generated/m2b/media/catalog.json \
  --index data/generated/m2b/vectors.json
```

The CLI loads the selected model only when a vector index is supplied. Model downloads require `--allow-download`; the same model and revision must be used to build and query an index.

In [ ]:
from savoroute.services.discovery import DiscoveryService
service = DiscoveryService(catalog)
via_service = service.rank("mushroom pasta", index=index, encoder=encoder,
                           catalog_dir=media_dir, filters=SearchFilters(kind="recipe"))
via_function = search_fused(catalog, "mushroom pasta", index=index, encoder=encoder,
                            catalog_dir=media_dir, filters=SearchFilters(kind="recipe"))
assert via_service == via_function
print("Service and retrieval API agree:", len(via_service), "results")
workspace.cleanup()